<a href="https://colab.research.google.com/github/kovalchukolena99-droid/mlops-class-2026/blob/main/PAF_Project_Group_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================================
# PAF Project Group 6
# Analysis of XXX
# ==========================================================================

In [ ]:
# Upload the Highway accidents data
from google.colab import files
uploaded = files.upload()

1. Upload the Highway accidents data


Saving highway_accidents.csv to highway_accidents.csv


In [ ]:
import pandas as pd
import numpy as np
import requests
import geopandas as gpd
import warnings
import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


In [ ]:
# Step 1. Inspecting raw data, its dimensions and column names
df = pd.read_csv("highway_accidents.csv")
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("Avaialble columns:")
print(df.columns.tolist())
print (df.head(3))

Dataset Shape: 150,000 rows x 23 columns
Avaialble columns:
['date', 'km', 'road_direction', 'roadway_type', 'area_type', 'latitude', 'longitude', 'time_of_day', 'highway_type', 'Uphill', 'Crossing', 'Curve', 'Downhill', 'Temporary_Detour', 'Roadworks', 'Road_Intersection', 'Bridge', 'Straight', 'Regulated_Uturn', 'Roundabout', 'Tunnel', 'Overpass', 'accident_classification']
         date    km  road_direction  roadway_type  area_type  latitude  \
0  2007-01-01  45.5             1.0           0.0        0.0       NaN   
1  2007-01-01  83.9             1.0           0.0        1.0       NaN   
2  2007-01-01  11.0             0.0           0.0        0.0       NaN   

   longitude  time_of_day  highway_type  Uphill  ...  Temporary_Detour  \
0        NaN            0           3.0     0.0  ...               0.0   
1        NaN            1           3.0     0.0  ...               0.0   
2        NaN            3           0.0     0.0  ...               0.0   

   Roadworks  Road_Intersec

In [ ]:
df["date"]=pd.to_datetime(df["date"])
df["year"]=df["date"].dt.year
df["month"]=df["date"].dt.month
df["day"]=df["date"].dt.day
df

,date,km,road_direction,roadway_type,area_type,latitude,longitude,time_of_day,highway_type,Uphill,...,Bridge,Straight,Regulated_Uturn,Roundabout,Tunnel,Overpass,accident_classification,year,month,day
0,2007-01-01,45.5,1.0,0.0,0.0,NaN,NaN,0,3.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0,2007,1,1
1,2007-01-01,83.9,1.0,0.0,1.0,NaN,NaN,1,3.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1,2007,1,1
2,2007-01-01,11.0,0.0,0.0,0.0,NaN,NaN,3,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0,2007,1,1
3,2007-01-01,180.0,1.0,2.0,1.0,NaN,NaN,0,4.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1,2007,1,1
4,2007-01-01,230.6,0.0,2.0,0.0,NaN,NaN,0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,2007,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,2025-12-31,412.5,0.0,0.0,1.0,-29.810654,-51.496686,3,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1,2025,12,31
149996,2025-12-31,51.0,1.0,0.0,0.0,-8.137602,-35.333387,2,4.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1,2025,12,31
149997,2025-12-31,645.5,1.0,0.0,0.0,-25.829749,-49.116659,0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0,2025,12,31
149998,2025-12-31,125.7,1.0,2.0,0.0,-23.969566,-50.235536,1,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1,2025,12,31


In [ ]:
#Step 2: Problem definition

#When an emergency call arrives, the dispatcher must chose between the standart
# patrol, basic ambulance, an immidiate Advanced Life Support or helicopter.
# Therefore, we chose to develop a model that estimates the probability of the crash being fatal.
# Hence, first step analysing the variables, in particular the outcome variable.

print("Inspecting 'accident_classification' value counts:")
acc_counts = df["accident_classification"].value_counts(dropna=False)
acc_pcts = (
    df["accident_classification"].value_counts(normalize=True, dropna=False)
    * 100)
target_profile = pd.DataFrame(
    {"Count": acc_counts, "Percentage (%)": acc_pcts.round(2)})
target_profile

# According to Data Dictionary: 0 = No Injuries, 1 = Injuries, 2 = Fatalities

target_profile["Data Dictionary Meaning"] = [
    "Injuries (Class 1)",
    "No Injuries (Class 0)",
    "Fatalities (Class 2)",
]
print(target_profile)

Inspecting 'accident_classification' value counts:
                         Count  Percentage (%) Data Dictionary Meaning
accident_classification                                               
1                        71781           47.85      Injuries (Class 1)
0                        70997           47.33   No Injuries (Class 0)
2                         7222            4.81    Fatalities (Class 2)


In [ ]:
#Since Class 2 of accident_classification is our target, the decision is to isolate it
df["is_fatal"] = (df["accident_classification"] == 2).astype(int)

print(f"\nTarget 'is_fatal':")
print(
    f"  - Non-Fatal (0): {(df['is_fatal'] == 0).sum():,} crashes ({ (1 - df['is_fatal'].mean())*100:.2f}%)"
)
print(
    f"  - Fatal (1):     {(df['is_fatal'] == 1).sum():,} crashes ({df['is_fatal'].mean()*100:.2f}%)"
)


Target 'is_fatal':
  - Non-Fatal (0): 142,778 crashes (95.19%)
  - Fatal (1):     7,222 crashes (4.81%)


In [ ]:
#Step 3: Data quality check

missing = df.isnull().mean()*100
print(missing.round(2).sort_values(ascending=False))


longitude                  70.82
latitude                   70.82
road_direction              0.06
km                          0.00
date                        0.00
area_type                   0.00
roadway_type                0.00
time_of_day                 0.00
highway_type                0.00
Uphill                      0.00
Crossing                    0.00
Curve                       0.00
Downhill                    0.00
Temporary_Detour            0.00
Roadworks                   0.00
Road_Intersection           0.00
Bridge                      0.00
Straight                    0.00
Regulated_Uturn             0.00
Roundabout                  0.00
Tunnel                      0.00
Overpass                    0.00
accident_classification     0.00
year                        0.00
month                       0.00
day                         0.00
is_fatal                    0.00
dtype: float64


In [ ]:
#From the first look we can see that around 70.82% of data in longitude and latitude
#are missing, while the rest of the fields have very little or no missing values.
# Next step is to analyse the reson why this data is missing and whether there is
# a particular pattern.

In [ ]:
pd.crosstab(df["year"], df["latitude"].isnull(), normalize="index")*100

latitude,False,True
year,,
2007,0.0,100.0
2008,0.0,100.0
2009,0.0,100.0
2010,0.0,100.0
2011,0.0,100.0
2012,0.0,100.0
2013,0.0,100.0
2014,0.0,100.0
2015,0.0,100.0


In [ ]:
# Further analysis uncovers that the GPS data is missing for 2007-2016.
# This is an important boundary that also shows that the GPS recordings only
# started from 2016.


In [ ]:
nulls_per_row = df.isnull().sum(axis=1)
corrupted_rows=df[nulls_per_row>10]
print(f"\nCorrupted Rows Found (>10 missing features): {len(corrupted_rows)}")


Corrupted Rows Found (>10 missing features): 1


In [ ]:
print("Corrupted Row Index:", corrupted_rows.index.tolist())
print("Sample Corrupted Row (Index 57163):",df.iloc[57163][
          ["date", "km", "Uphill", "Curve", "Bridge", "accident_classification"]
      ].to_dict(),)

Corrupted Row Index: [57163]
Sample Corrupted Row (Index 57163): {'date': Timestamp('2012-03-10 00:00:00'), 'km': 87.1, 'Uphill': nan, 'Curve': nan, 'Bridge': nan, 'accident_classification': 0}



Class Distribution Across Key Year Groups (Reporting Shift):
2007 Sample:
 Class 0 (No Injury %)    59.5
Class 1 (Injury %)       36.7
Class 2 (Fatality %)      3.8
Name: 2007, dtype: float64
2025 Sample:
 Class 0 (No Injury %)    15.5
Class 1 (Injury %)       77.5
Class 2 (Fatality %)      7.0
Name: 2025, dtype: float64
